Michael Tyler
DATA 606


## Data Loader
This will load 7 data sources:

* IHME United States Mortality Rates and Life Expectancy by County,Race, and Ethnicity 2000-2019
* County Health Rankings 2020 national analytic data 
* USDA ERS Rural-Urban Continuum Codes, 2013 version 
* CMS Medicare Geographic Variation by National, State & County (CSV)
* ACS Census ACS 5-year 2015-2019, downloaded via the API.
* IRS County Migration data. Inflow and outflow
* Social Capital Atlas

In [1]:
  pip install pandas numpy requests openpyxl xlrd

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#!/usr/bin/env python3
"""
build_dataset.py
----------------
Builds a county-level dataset for modeling life expectancy at age 50 (2019).

WHERE TO PUT THE FILES
Download these by hand into the folders shown. File names don't matter; the
script looks for any matching file in each folder. The ACS data is pulled
from the Census API automatically.

  data/raw/ihme/  IHME "United States Mortality Rates and Life Expectancy by County,
                  Race, and Ethnicity 2000-2019" (unzip the CSV files here)
                  https://ghdx.healthdata.org/record/ihme-data/united-states-life-expectancy-by-county-race-ethnicity-2000-2019
  data/raw/chr/   County Health Rankings 2020 national analytic data (CSV)
                  https://www.countyhealthrankings.org/health-data/methodology-and-sources/data-documentation/national-data-documentation-2010-2023
  data/raw/rucc/  USDA ERS Rural-Urban Continuum Codes, 2013 version (xls/xlsx/csv)
                  https://ers.usda.gov/data-products/rural-urban-continuum-codes
  data/raw/cms/   CMS Medicare Geographic Variation by National, State & County (CSV)
                  https://data.cms.gov/summary-statistics-on-use-and-payments/medicare-geographic-comparisons/medicare-geographic-variation-by-national-state-county
  ACS             Census ACS 5-year 2015-2019, downloaded via the API.
                  Get a free key: https://api.census.gov/data/key_signup.html



Only the IHME outcome file is required. Any other source whose folder is
empty is skipped with a warning, so you can add sources one at a time.
"""

from __future__ import annotations

import argparse
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------
OUTCOME_YEAR = 2019      # IHME estimates end in 2019
ACS_YEAR = 2019          # ACS 5-year 2015-2019
CMS_YEAR = 2019
RUCC_COLUMN = "RUCC_2013"

# County FIPS codes that changed; old code -> current code.
# Applied to every source so they all join on the same code.
FIPS_RECODES = {
    "46113": "46102",  # Shannon County, SD renamed Oglala Lakota County (2015)
    "02270": "02158",  # Wade Hampton Census Area, AK renamed Kusilvak (2015)
    "51515": "51019",  # Bedford city, VA merged into Bedford County (2013)
}

STATE_ABBR = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA", "08": "CO",
    "09": "CT", "10": "DE", "11": "DC", "12": "FL", "13": "GA", "15": "HI",
    "16": "ID", "17": "IL", "18": "IN", "19": "IA", "20": "KS", "21": "KY",
    "22": "LA", "23": "ME", "24": "MD", "25": "MA", "26": "MI", "27": "MN",
    "28": "MS", "29": "MO", "30": "MT", "31": "NE", "32": "NV", "33": "NH",
    "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND", "39": "OH",
    "40": "OK", "41": "OR", "42": "PA", "44": "RI", "45": "SC", "46": "SD",
    "47": "TN", "48": "TX", "49": "UT", "50": "VT", "51": "VA", "53": "WA",
    "54": "WV", "55": "WI", "56": "WY",
}

# Census divisions, useful for spatial cross-validation later
DIVISIONS = {
    "New England": ["CT", "ME", "MA", "NH", "RI", "VT"],
    "Middle Atlantic": ["NJ", "NY", "PA"],
    "East North Central": ["IL", "IN", "MI", "OH", "WI"],
    "West North Central": ["IA", "KS", "MN", "MO", "NE", "ND", "SD"],
    "South Atlantic": ["DE", "DC", "FL", "GA", "MD", "NC", "SC", "VA", "WV"],
    "East South Central": ["AL", "KY", "MS", "TN"],
    "West South Central": ["AR", "LA", "OK", "TX"],
    "Mountain": ["AZ", "CO", "ID", "MT", "NV", "NM", "UT", "WY"],
    "Pacific": ["AK", "CA", "HI", "OR", "WA"],
}
STATE_TO_DIVISION = {s: d for d, states in DIVISIONS.items() for s in states}

# CHR measures that are built from death data. Using them as predictors
# would leak the outcome into the model, so they are dropped.
CHR_LEAKAGE = re.compile(
    r"premature|life expectancy|mortality|death|fatalit|"
    r"years of potential life|suicide|homicide",
    re.IGNORECASE,
)

# ACS detailed-table variables (stable IDs across years)
ACS_VARS = {
    "B01003_001E": "total_pop",
    "B01002_001E": "median_age",
    "B19013_001E": "median_hh_income",
    "B19083_001E": "gini",
    "B25077_001E": "median_home_value",
    "B17001_001E": "pov_universe",
    "B17001_002E": "pov_below",
    "B23025_003E": "civ_labor_force",
    "B23025_005E": "unemployed",
    "B15003_001E": "edu_universe",
    "B03002_001E": "race_universe",
    "B03002_003E": "nh_white",
    "B03002_004E": "nh_black",
    "B03002_005E": "nh_aian",
    "B03002_006E": "nh_asian",
    "B03002_012E": "hispanic",
    "B05002_001E": "nativity_universe",
    "B05002_013E": "foreign_born",
}
# Educational attainment, age 25+: 017-025 = HS diploma or higher,
# 022-025 = bachelor's or higher
for i in range(17, 26):
    ACS_VARS[f"B15003_0{i}E"] = f"edu_{i}"

# CMS variables to keep if present (names can vary between releases;
# the script reports any it can't find)
CMS_VARS = {
    "BENE_AVG_AGE": ("cms_avg_bene_age", "Average age of Medicare FFS beneficiaries"),
    "BENE_DUAL_PCT": ("cms_pct_dual_eligible", "% of FFS beneficiaries also eligible for Medicaid"),
    "MA_PRTCPTN_RATE": ("cms_ma_participation_rate", "Medicare Advantage participation rate"),
    "TOT_MDCR_STDZD_PYMT_PC": ("cms_std_spending_per_capita", "Standardized Medicare spending per FFS beneficiary"),
    "BENE_AVG_RISK_SCRE": ("cms_avg_risk_score", "Average HCC risk score (CAUTION: measures illness, close to the outcome)"),
    "ER_VISITS_PER_1000_BENES": ("cms_er_visits_per_1000", "ER visits per 1,000 FFS beneficiaries"),
    "ACUTE_HOSP_READMSN_PCT": ("cms_readmission_pct", "30-day hospital readmission %"),
}

print("finished")

finished


## Change Working Directory

In [5]:
# Find current working directory
print("Current Directory:", os.getcwd())
os.chdir(r"C:\Users\14438")
print("New Directory:", os.getcwd())


Current Directory: C:\Users\14438
New Directory: C:\Users\14438


## Must have Census API Key To Download.

In [6]:
import os
os.environ["CENSUS_API_KEY"] = "38ee56684fff466ad7ea656bac74976d9022ab45"


## Check to Make sure all the files are there

In [7]:

from pathlib import Path

RAW = Path("data/raw")            # relative to the working directory set above
NEEDED = {
    "ihme": "IHME life expectancy 2015-2019 CSV",
    "chr":  "County Health Rankings 2020 analytic data CSV",
    "rucc": "USDA Rural-Urban Continuum Codes 2013",
    "cms":  "CMS Medicare Geographic Variation county CSV",
    "irs":  "IRS countyinflow + countyoutflow CSVs",
    "scap": "Social Capital Atlas social_capital_county.csv",
    "acs":  "(downloaded automatically from the Census API)",
}
for folder, what in NEEDED.items():
    path = RAW / folder
    path.mkdir(parents=True, exist_ok=True)
    files = [f.name for f in path.iterdir() if f.is_file()]
    status = ", ".join(files) if files else f"EMPTY - add {what}"
    print(f"{folder:6} {status}")
print(f"\nFolders are under: {RAW.resolve()}")

ihme   IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2015_BOTH_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2015_FEMALE_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2015_MALE_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2016_BOTH_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2016_FEMALE_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2016_MALE_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2017_BOTH_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2017_FEMALE_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2017_MALE_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2018_BOTH_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2018_FEMALE_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2018_MALE_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2019_BOTH_Y2022M06D16.CSV, IHME_USA_LE_COUNTY_RACE_ETHN_2000_2019_LT_2019_FEMALE_Y2022M06D16.CSV, IHME_USA_LE_COUNT

## Process all the files and create measures

In [13]:

"""

----------------
Builds a county-level dataset for modeling life expectancy at age 50 (2019).

WHERE TO PUT THE FILES
Download these by hand into the folders shown. File names don't matter; the
script looks for any matching file in each folder. The ACS data is pulled
from the Census API automatically.

  data/raw/ihme/  IHME "United States Mortality Rates and Life Expectancy by County,
                  Race, and Ethnicity 2000-2019" (unzip the CSV files here)
                  https://ghdx.healthdata.org/record/ihme-data/united-states-life-expectancy-by-county-race-ethnicity-2000-2019
  data/raw/chr/   County Health Rankings 2020 national analytic data (CSV)
                  https://www.countyhealthrankings.org/health-data/methodology-and-sources/data-documentation/national-data-documentation-2010-2023
  data/raw/rucc/  USDA ERS Rural-Urban Continuum Codes, 2013 version (xls/xlsx/csv)
                  https://ers.usda.gov/data-products/rural-urban-continuum-codes
  data/raw/cms/   CMS Medicare Geographic Variation by National, State & County (CSV)
                  https://data.cms.gov/summary-statistics-on-use-and-payments/medicare-geographic-comparisons/medicare-geographic-variation-by-national-state-county
  data/raw/irs/   IRS SOI county-to-county migration, BOTH files for one year pair:
                  countyinflow1718.csv and countyoutflow1718.csv (or 1819)
                  https://www.irs.gov/statistics/soi-tax-stats-migration-data
  data/raw/scap/  Opportunity Insights / Meta Social Capital Atlas, county file
                  (social_capital_county.csv)
                  https://data.humdata.org/dataset/social-capital-atlas
  ACS             Census ACS 5-year 2015-2019, downloaded via the API.
                  Get a free key: https://api.census.gov/data/key_signup.html

HOW TO RUN
  pip install pandas numpy requests openpyxl xlrd
  export CENSUS_API_KEY=your_key        (Windows: set CENSUS_API_KEY=your_key)
  python build_dataset.py

OUTPUTS (data/processed/)
  county_le50_2019.csv     one row per county: outcomes + predictors
  data_dictionary.csv      column name, source, and description
  missingness_report.csv   share of missing values in each column

When loading the output, keep FIPS codes as text so leading zeros survive:
  pd.read_csv("data/processed/county_le50_2019.csv",
              dtype={"fips": str, "state_fips": str})

Only the IHME outcome file is required. Any other source whose folder is
empty is skipped with a warning, so you can add sources one at a time.
"""

from __future__ import annotations

import argparse
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# ---------------------------------------------------------------------------
# Settings
# ---------------------------------------------------------------------------
OUTCOME_YEAR = 2019      # IHME estimates end in 2019
ACS_YEAR = 2019          # ACS 5-year 2015-2019
CMS_YEAR = 2019
RUCC_COLUMN = "RUCC_2013"

# County FIPS codes that changed; old code -> code used by the 2019 predictors.
# Applied to every source so they all join on the same code. If a source already
# has the target code, the old row is dropped instead of renamed.
FIPS_RECODES = {
    "46113": "46102",  # Shannon County, SD renamed Oglala Lakota County (2015)
    "02270": "02158",  # Wade Hampton Census Area, AK renamed Kusilvak (2015)
    "51515": "51019",  # Bedford city, VA merged into Bedford County (2013)
    "12025": "12086",  # Dade County, FL renamed Miami-Dade County (1997)
    "51560": "51005",  # Clifton Forge city, VA merged into Alleghany County (2001)
    "51780": "51083",  # South Boston city, VA merged into Halifax County (1995)
    "46131": "46071",  # Washabaugh County, SD merged into Jackson County (1983)
    # Valdez-Cordova, AK (02261) split into Chugach (02063) and Copper River
    # (02066) in 2019. 2019 predictors still use 02261, so map the new codes back.
    "02063": "02261",
    "02066": "02261",
}

# Outcome columns (used to find counties IHME combined into one unit)
OUTCOME_COLS = ["le50_both", "le50_male", "le50_female", "le0_both"]
# Predictors that are counts: summed (not averaged) when combining counties
SUM_COLS = {"acs_total_pop", "chr_population", "irs_in_returns", "irs_out_returns",
            "irs_nonmigrant_returns"}
# Predictors taken from the largest county when combining (categorical codes)
LARGEST_COLS = {"rucc", "metro"}

STATE_ABBR = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA", "08": "CO",
    "09": "CT", "10": "DE", "11": "DC", "12": "FL", "13": "GA", "15": "HI",
    "16": "ID", "17": "IL", "18": "IN", "19": "IA", "20": "KS", "21": "KY",
    "22": "LA", "23": "ME", "24": "MD", "25": "MA", "26": "MI", "27": "MN",
    "28": "MS", "29": "MO", "30": "MT", "31": "NE", "32": "NV", "33": "NH",
    "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND", "39": "OH",
    "40": "OK", "41": "OR", "42": "PA", "44": "RI", "45": "SC", "46": "SD",
    "47": "TN", "48": "TX", "49": "UT", "50": "VT", "51": "VA", "53": "WA",
    "54": "WV", "55": "WI", "56": "WY",
}

# Census divisions, useful for spatial cross-validation later
DIVISIONS = {
    "New England": ["CT", "ME", "MA", "NH", "RI", "VT"],
    "Middle Atlantic": ["NJ", "NY", "PA"],
    "East North Central": ["IL", "IN", "MI", "OH", "WI"],
    "West North Central": ["IA", "KS", "MN", "MO", "NE", "ND", "SD"],
    "South Atlantic": ["DE", "DC", "FL", "GA", "MD", "NC", "SC", "VA", "WV"],
    "East South Central": ["AL", "KY", "MS", "TN"],
    "West South Central": ["AR", "LA", "OK", "TX"],
    "Mountain": ["AZ", "CO", "ID", "MT", "NV", "NM", "UT", "WY"],
    "Pacific": ["AK", "CA", "HI", "OR", "WA"],
}
STATE_TO_DIVISION = {s: d for d, states in DIVISIONS.items() for s in states}

# CHR measures that are built from death data. Using them as predictors
# would leak the outcome into the model, so they are dropped.
CHR_LEAKAGE = re.compile(
    r"premature|life expectancy|mortality|death|fatalit|"
    r"years of potential life|suicide|homicide",
    re.IGNORECASE,
)
# Cause-specific death measures to KEEP despite the filter above (by choice).
# They overlap with the outcome, so they're flagged in the data dictionary and
# listed in MORTALITY_PREDICTORS for easy removal in a "without" model.
CHR_KEEP_ANYWAY = [
    "injury deaths",
    "drug overdose deaths",
    "homicides",
    "suicides",
    "firearm fatalities",
    "motor vehicle crash deaths",
]

# ACS detailed-table variables (stable IDs across years)
ACS_VARS = {
    "B01003_001E": "total_pop",
    "B01002_001E": "median_age",
    "B19013_001E": "median_hh_income",
    "B19083_001E": "gini",
    "B25077_001E": "median_home_value",
    "B17001_001E": "pov_universe",
    "B17001_002E": "pov_below",
    "B23025_003E": "civ_labor_force",
    "B23025_005E": "unemployed",
    "B15003_001E": "edu_universe",
    "B03002_001E": "race_universe",
    "B03002_003E": "nh_white",
    "B03002_004E": "nh_black",
    "B03002_005E": "nh_aian",
    "B03002_006E": "nh_asian",
    "B03002_012E": "hispanic",
    "B05002_001E": "nativity_universe",
    "B05002_013E": "foreign_born",
}
# Educational attainment, age 25+: 017-025 = HS diploma or higher,
# 022-025 = bachelor's or higher
for i in range(17, 26):
    ACS_VARS[f"B15003_0{i}E"] = f"edu_{i}"

# CMS variables to keep if present (names can vary between releases;
# the script reports any it can't find)
CMS_VARS = {
    "BENE_AVG_AGE": ("cms_avg_bene_age", "Average age of Medicare FFS beneficiaries"),
    "BENE_DUAL_PCT": ("cms_pct_dual_eligible", "% of FFS beneficiaries also eligible for Medicaid"),
    "MA_PRTCPTN_RATE": ("cms_ma_participation_rate", "Medicare Advantage participation rate"),
    "TOT_MDCR_STDZD_PYMT_PC": ("cms_std_spending_per_capita", "Standardized Medicare spending per FFS beneficiary"),
    "BENE_AVG_RISK_SCRE": ("cms_avg_risk_score", "Average HCC risk score (CAUTION: measures illness, close to the outcome)"),
    "ER_VISITS_PER_1000_BENES": ("cms_er_visits_per_1000", "ER visits per 1,000 FFS beneficiaries"),
    "ACUTE_HOSP_READMSN_PCT": ("cms_readmission_pct", "30-day hospital readmission %"),
}


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def log(msg: str) -> None:
    print(msg, flush=True)


def find_files(folder: Path, exts: tuple[str, ...]) -> list[Path]:
    if not folder.exists():
        return []
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in exts)


def clean_fips(s: pd.Series) -> pd.Series:
    """Turn 1001, '1001', '1001.0', or '01001' into '01001'."""
    s = s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    return s.str.extract(r"^(\d{1,5})$", expand=False).str.zfill(5)


def is_county_fips(f) -> bool:
    return (
        isinstance(f, str)
        and len(f) == 5
        and not f.endswith("000")
        and f[:2] in STATE_ABBR
    )


def harmonize_fips(df: pd.DataFrame, source: str) -> pd.DataFrame:
    df = df.copy()
    for old, new in FIPS_RECODES.items():
        old_rows = df["fips"] == old
        if old_rows.any():
            if (df["fips"] == new).any():
                df = df[~old_rows]          # new code already present; drop old
            else:
                df.loc[old_rows, "fips"] = new
    dups = df["fips"].duplicated(keep="first")
    if dups.any():
        log(f"  [{source}] WARNING: {dups.sum()} duplicate FIPS rows; keeping the first of each")
        df = df[~dups]
    return df


def slug(text: str, max_len: int = 45) -> str:
    text = re.sub(r"raw value", "", str(text), flags=re.IGNORECASE)
    text = text.replace("%", "pct ")
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")[:max_len]


def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() in (".xls", ".xlsx"):
        return pd.read_excel(path, dtype=str)
    try:
        return pd.read_csv(path, dtype=str, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, dtype=str, low_memory=False, encoding="latin-1")


# ---------------------------------------------------------------------------
# Outcome: IHME life expectancy
# ---------------------------------------------------------------------------
IHME_COLS = {
    "fips": ["fips", "location_fips", "fips_code"],
    "year": ["year", "year_id"],
    "sex": ["sex", "sex_name", "sex_label"],
    "age": ["age", "age_name", "age_group_name", "age_group"],
    "race": ["race", "race_name", "race_label", "race_ethnicity"],
    "measure": ["measure", "measure_name"],
    "value": ["val", "mean", "value"],
}
AGE50 = r"^50\b"                           # e.g. "50 to 54"
AGE0 = r"^(<\s*1|0\b|under 1|birth)"      # e.g. "<1 year"


def _pick(columns, candidates):
    lower = {c.lower(): c for c in columns}
    return next((lower[c] for c in candidates if c in lower), None)


def _show_ihme_values(files: list[Path]) -> None:
    """Print the values in each column so filters can be adjusted."""
    sample = pd.read_csv(files[0], nrows=200_000, dtype=str)
    log(f"  Columns in {files[0].name}: {list(sample.columns)}")
    for key, cands in IHME_COLS.items():
        col = _pick(sample.columns, cands)
        if col and key in ("sex", "age", "race", "measure", "year"):
            log(f"  {col}: {sorted(sample[col].dropna().unique())[:30]}")


def load_ihme(folder: Path):
    files = find_files(folder, (".csv",))
    if not files:
        sys.exit(f"ERROR: no IHME CSV files found in {folder}. This outcome file is required.")

    parts = []
    for f in files:
        header = pd.read_csv(f, nrows=0).columns
        col = {k: _pick(header, v) for k, v in IHME_COLS.items()}
        missing = [k for k in ("fips", "year", "age", "value") if col[k] is None]
        if missing:
            log(f"  [IHME] skipping {f.name}: can't find columns for {missing}")
            continue
        usecols = [c for c in col.values() if c]
        rename = {v: k for k, v in col.items() if v}
        for chunk in pd.read_csv(f, usecols=usecols, dtype=str, chunksize=500_000):
            chunk = chunk.rename(columns=rename)
            keep = pd.to_numeric(chunk["year"], errors="coerce") == OUTCOME_YEAR
            if "measure" in chunk:
                keep &= chunk["measure"].str.contains("life expectancy", case=False, na=False)
            if "race" in chunk:
                keep &= chunk["race"].str.strip().str.lower().str.match(r"^(total|all)", na=False)
            age = chunk["age"].str.strip().str.lower()
            is50 = age.str.match(AGE50, na=False)
            is0 = age.str.match(AGE0, na=False)
            keep &= is50 | is0
            if keep.any():
                sub = chunk[keep].copy()
                sub["age_key"] = np.where(is50[keep], "le50", "le0")
                parts.append(sub)

    if not parts:
        log("ERROR: no IHME rows matched the filters (year, life expectancy, total race, age 0/50).")
        _show_ihme_values(files)
        sys.exit("Adjust the filters in load_ihme() to match the values printed above.")

    df = pd.concat(parts, ignore_index=True)
    if "sex" not in df:
        df["sex"] = "both"
    sex = df["sex"].str.strip().str.lower()
    df["sex_key"] = np.select(
        [sex.str.startswith("both"), sex.str.startswith("female"), sex.str.startswith("male")],
        ["both", "female", "male"],
        default="other",
    )
    df["fips"] = clean_fips(df["fips"])
    df = df[df["fips"].map(is_county_fips)]
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df["outcome"] = df["age_key"] + "_" + df["sex_key"]

    if df.duplicated(["fips", "outcome"]).any():
        log("  [IHME] WARNING: multiple rows per county/outcome (check race or measure filters); keeping the first")

    wide = df.pivot_table(index="fips", columns="outcome", values="value", aggfunc="first").reset_index()
    wide.columns.name = None
    wanted = ["le50_both", "le50_male", "le50_female", "le0_both"]
    if "le50_both" not in wide:
        _show_ihme_values(files)
        sys.exit("ERROR: couldn't build le50_both (both sexes, age 50). Check the values printed above.")
    wide = wide[["fips"] + [c for c in wanted if c in wide]]

    descriptions = {
        "le50_both": "OUTCOME: life expectancy at age 50, both sexes (IHME, 2019)",
        "le50_male": "Life expectancy at age 50, males (IHME, 2019)",
        "le50_female": "Life expectancy at age 50, females (IHME, 2019)",
        "le0_both": "Life expectancy at birth, both sexes (IHME, 2019), for comparison",
    }
    entries = [{"column": c, "source": "IHME", "description": descriptions[c]} for c in wide.columns if c != "fips"]
    log(f"  [IHME] {len(wide):,} counties; outcomes: {[c for c in wide.columns if c != 'fips']}")
    return wide, entries


# ---------------------------------------------------------------------------
# County Health Rankings
# ---------------------------------------------------------------------------
def load_chr(folder: Path):
    files = find_files(folder, (".csv",))
    if not files:
        if find_files(folder, (".xls", ".xlsx")):
            log(f"  [CHR] found only an Excel workbook in {folder}. This script uses the single-table "
                "'2020 CHR Analytic Data (CSV)' file (analytic_data2020_0.csv) from the same CHR page; skipping")
        else:
            log(f"  [CHR] no CSV in {folder}; skipping")
        return None, []
    path = next((f for f in files if "2020" in f.name), files[0])

    # CHR analytic files have a row of readable labels above the variable-name row
    top = pd.read_csv(path, header=None, nrows=2, dtype=str)
    row0 = top.iloc[0].str.lower().tolist()
    row1 = top.iloc[1].str.lower().tolist()
    if "fipscode" in row1:
        labels = top.iloc[0].tolist()
        df = pd.read_csv(path, header=1, dtype=str, low_memory=False)
    elif "fipscode" in row0:
        labels = top.iloc[0].tolist()
        df = pd.read_csv(path, header=0, dtype=str, low_memory=False)
    else:
        log(f"  [CHR] can't find a 'fipscode' column in {path.name}; skipping")
        return None, []
    df.columns = [c.lower() for c in df.columns]
    label_map = dict(zip(df.columns, labels))

    df = df[df["countycode"].str.zfill(3) != "000"].copy()   # drop state/national rows
    df["fips"] = clean_fips(df["fipscode"])

    raw_cols = [c for c in df.columns if c.endswith("_rawvalue")]
    keep, dropped, kept_mortality = [], [], []
    for c in raw_cols:
        label = str(label_map.get(c, "")).lower()
        exempt = any(k in label for k in CHR_KEEP_ANYWAY)
        if CHR_LEAKAGE.search(label) and not exempt:
            dropped.append(c)
        else:
            keep.append(c)
            if exempt:
                kept_mortality.append(c)

    out = df[["fips"] + keep].copy()
    out[keep] = out[keep].apply(pd.to_numeric, errors="coerce")

    names, entries = {}, []
    for c in keep:
        label = str(label_map.get(c, c))
        name = "chr_" + slug(label)
        while name in names.values():
            name += "_2"
        names[c] = name
        desc = label + (" [MORTALITY-BASED: overlaps outcome]" if c in kept_mortality else "")
        entries.append({"column": name, "source": f"CHR ({path.name}, {c})", "description": desc})
    out = out.rename(columns=names)

    log(f"  [CHR] {len(out):,} counties, {len(keep)} measures kept")
    if dropped:
        log(f"  [CHR] dropped {len(dropped)} death-based measures to avoid leakage: "
            f"{[label_map.get(c) for c in dropped]}")
    if kept_mortality:
        log(f"  [CHR] kept {len(kept_mortality)} cause-specific death measures by choice: "
            f"{[names[c] for c in kept_mortality]}")
    return out, entries


# ---------------------------------------------------------------------------
# Census ACS (API)
# ---------------------------------------------------------------------------
def load_acs(cache: Path):
    if cache.exists():
        log(f"  [ACS] using cached file {cache}")
        df = pd.read_csv(cache, dtype=str)
    else:
        key = os.environ.get("CENSUS_API_KEY")
        if not key:
            log("  [ACS] CENSUS_API_KEY not set; skipping ACS")
            return None, []
        url = (f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5"
               f"?get=NAME,{','.join(ACS_VARS)}&for=county:*&key={key}")
        log("  [ACS] downloading from the Census API...")
        r = requests.get(url, timeout=180)
        if r.status_code != 200:
            log(f"  [ACS] API error {r.status_code}: {r.text[:300]}")
            return None, []
        rows = r.json()
        df = pd.DataFrame(rows[1:], columns=rows[0])
        cache.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(cache, index=False)
        log(f"  [ACS] saved raw download to {cache}")

    df["fips"] = df["state"].str.zfill(2) + df["county"].str.zfill(3)
    a = df[list(ACS_VARS)].apply(pd.to_numeric, errors="coerce")
    a = a.mask(a <= -222222222)          # Census codes for missing values
    a = a.rename(columns=ACS_VARS)

    def pct(num, den):
        return (100 * num / den).replace([np.inf, -np.inf], np.nan)

    hs = a[[f"edu_{i}" for i in range(17, 26)]].sum(axis=1, min_count=1)
    ba = a[[f"edu_{i}" for i in range(22, 26)]].sum(axis=1, min_count=1)

    out = pd.DataFrame({"fips": df["fips"]})
    specs = {
        "acs_total_pop": (a["total_pop"], "Total population"),
        "acs_log_pop": (np.log(a["total_pop"].where(a["total_pop"] > 0)), "Log of total population"),
        "acs_median_age": (a["median_age"], "Median age"),
        "acs_median_hh_income": (a["median_hh_income"], "Median household income (2019 $)"),
        "acs_gini": (a["gini"], "Gini index of income inequality"),
        "acs_median_home_value": (a["median_home_value"], "Median home value (2019 $)"),
        "acs_pct_poverty": (pct(a["pov_below"], a["pov_universe"]), "% of people below poverty"),
        "acs_unemployment_rate": (pct(a["unemployed"], a["civ_labor_force"]), "% of civilian labor force unemployed"),
        "acs_pct_hs_or_higher": (pct(hs, a["edu_universe"]), "% age 25+ with HS diploma/GED or higher"),
        "acs_pct_bachelors_or_higher": (pct(ba, a["edu_universe"]), "% age 25+ with bachelor's degree or higher"),
        "acs_pct_nh_white": (pct(a["nh_white"], a["race_universe"]), "% non-Hispanic White"),
        "acs_pct_nh_black": (pct(a["nh_black"], a["race_universe"]), "% non-Hispanic Black"),
        "acs_pct_nh_aian": (pct(a["nh_aian"], a["race_universe"]), "% non-Hispanic American Indian/Alaska Native"),
        "acs_pct_nh_asian": (pct(a["nh_asian"], a["race_universe"]), "% non-Hispanic Asian"),
        "acs_pct_hispanic": (pct(a["hispanic"], a["race_universe"]), "% Hispanic or Latino"),
        "acs_pct_foreign_born": (pct(a["foreign_born"], a["nativity_universe"]), "% foreign born"),
    }
    entries = []
    for name, (values, desc) in specs.items():
        out[name] = values
        entries.append({"column": name, "source": f"Census ACS 5-year {ACS_YEAR}", "description": desc})

    out = out[out["fips"].map(is_county_fips)]
    log(f"  [ACS] {len(out):,} counties, {len(specs)} variables")
    return out, entries


# ---------------------------------------------------------------------------
# USDA Rural-Urban Continuum Codes
# ---------------------------------------------------------------------------
def load_rucc(folder: Path):
    files = find_files(folder, (".xls", ".xlsx", ".csv"))
    if not files:
        log(f"  [RUCC] no file in {folder}; skipping")
        return None, []
    df = read_table(files[0])
    lower = {c.lower(): c for c in df.columns}
    fips_col = next((lower[c] for c in lower if "fips" in c), None)
    if fips_col is None:
        log(f"  [RUCC] no FIPS column in {files[0].name}; skipping")
        return None, []

    if "attribute" in lower and "value" in lower:           # long format (2023 style)
        attr, val = lower["attribute"], lower["value"]
        rucc_rows = df[df[attr].str.upper().str.startswith("RUCC")]
        df = rucc_rows.rename(columns={val: "rucc"})
    else:                                                    # wide format (2013 style)
        rucc_col = lower.get(RUCC_COLUMN.lower()) or next(
            (lower[c] for c in lower if c.startswith("rucc")), None)
        if rucc_col is None:
            log(f"  [RUCC] no RUCC column in {files[0].name}; skipping")
            return None, []
        if rucc_col.lower() != RUCC_COLUMN.lower():
            log(f"  [RUCC] note: using {rucc_col} (expected {RUCC_COLUMN})")
        df = df.rename(columns={rucc_col: "rucc"})

    out = pd.DataFrame({
        "fips": clean_fips(df[fips_col]),
        "rucc": pd.to_numeric(df["rucc"], errors="coerce"),
    })
    out = out[out["fips"].map(is_county_fips)].dropna(subset=["rucc"])
    out["metro"] = (out["rucc"] <= 3).astype(int)
    entries = [
        {"column": "rucc", "source": f"USDA ERS ({files[0].name})",
         "description": "Rural-Urban Continuum Code: 1-3 metro (by size), 4-9 nonmetro (by urbanization/adjacency)"},
        {"column": "metro", "source": "Derived from rucc", "description": "1 = metro county (RUCC 1-3)"},
    ]
    log(f"  [RUCC] {len(out):,} counties")
    return out, entries


# ---------------------------------------------------------------------------
# CMS Medicare Geographic Variation
# ---------------------------------------------------------------------------
def load_cms(folder: Path):
    files = find_files(folder, (".csv",))
    if not files:
        log(f"  [CMS] no CSV in {folder}; skipping")
        return None, []
    df = read_table(files[0])
    upper = {c.upper(): c for c in df.columns}
    needed = ["BENE_GEO_LVL", "BENE_GEO_CD", "YEAR"]
    if any(n not in upper for n in needed):
        log(f"  [CMS] expected columns {needed} not found. Columns are: {list(df.columns)[:25]}...")
        return None, []

    rows = (df[upper["BENE_GEO_LVL"]].str.strip().str.lower() == "county") & \
           (pd.to_numeric(df[upper["YEAR"]], errors="coerce") == CMS_YEAR)
    if "BENE_AGE_LVL" in upper:
        rows &= df[upper["BENE_AGE_LVL"]].str.strip().str.lower() == "all"
    df = df[rows]
    if df.empty:
        log(f"  [CMS] no county rows for {CMS_YEAR}; check that the file includes that year")
        return None, []

    out = pd.DataFrame({"fips": clean_fips(df[upper["BENE_GEO_CD"]])})
    entries, missing = [], []
    for src, (name, desc) in CMS_VARS.items():
        if src in upper:
            # '*' marks suppressed values; to_numeric turns them into NaN
            out[name] = pd.to_numeric(df[upper[src]].str.replace(",", ""), errors="coerce").values
            entries.append({"column": name, "source": f"CMS Geographic Variation {CMS_YEAR} ({src})", "description": desc})
        else:
            missing.append(src)
    if missing:
        log(f"  [CMS] not found (names may differ in your release; see the data dictionary): {missing}")
    out = out[out["fips"].map(is_county_fips)]
    log(f"  [CMS] {len(out):,} counties, {len(entries)} variables")
    return out, entries


# ---------------------------------------------------------------------------
# IRS county-to-county migration
# ---------------------------------------------------------------------------
# Inflow file:  y2_* = this county (destination), y1_* = where movers came from
# Outflow file: y1_* = this county (origin),      y2_* = where movers went
# Summary rows use pseudo state codes 57-98 (totals, foreign, non-migrants) and
# county code 000; only real county-to-county flows (state 01-56) are summed.
# ---------------------------------------------------------------------------
# IRS county-to-county migration
# ---------------------------------------------------------------------------
# Inflow file:  y2_* = this county (destination), y1_* = where movers came from
# Outflow file: y1_* = this county (origin),      y2_* = where movers went
# Summary rows use pseudo state codes 57-98 (totals, foreign, non-migrants) and
# county code 000; only real county-to-county flows (state 01-56) are summed.
def load_irs(folder: Path):
    files = find_files(folder, (".csv",))
    if not files:
        log(f"  [IRS] no CSV in {folder}; skipping")
        return None, []

    frames = {}
    for path in files:
        # Some years contain non-UTF-8 county names (e.g. Dona Ana), so fall back
        # to latin-1 rather than failing on the whole file.
        def read_irs(**kw):
            try:
                return pd.read_csv(path, dtype=str, low_memory=False, **kw)
            except UnicodeDecodeError:
                return pd.read_csv(path, dtype=str, low_memory=False, encoding="latin-1", **kw)

        actual = {c.lower(): c for c in read_irs(nrows=0).columns}
        header = list(actual)
        name = path.name.lower()
        if "inflow" in name:
            kind, self_pre, other_pre = "in", "y2", "y1"
        elif "outflow" in name:
            kind, self_pre, other_pre = "out", "y1", "y2"
        else:
            log(f"  [IRS] skipping {path.name}: name says neither inflow nor outflow")
            continue
        need = [f"{self_pre}_statefips", f"{self_pre}_countyfips",
                f"{other_pre}_statefips", f"{other_pre}_countyfips", "n1", "agi"]
        if any(c not in header for c in need):
            log(f"  [IRS] skipping {path.name}: expected columns {need}, found {header}")
            continue

        df = read_irs(usecols=[actual[c] for c in need])
        df.columns = [c.lower() for c in df.columns]
        df["fips"] = (df[f"{self_pre}_statefips"].str.zfill(2)
                      + df[f"{self_pre}_countyfips"].str.zfill(3))
        df = df[df["fips"].map(is_county_fips)].copy()
        df["n1"] = pd.to_numeric(df["n1"], errors="coerce")
        df["agi"] = pd.to_numeric(df["agi"], errors="coerce")   # thousands of dollars

        other_state = pd.to_numeric(df[f"{other_pre}_statefips"], errors="coerce")
        other_county = df[f"{other_pre}_countyfips"].str.zfill(3)
        other_fips = df[f"{other_pre}_statefips"].str.zfill(2) + other_county

        # Each county has summary rows using pseudo state codes:
        #   97/000 = Total Migration-US (all domestic moves, INCLUDING the small
        #            flows that are suppressed individually and pooled into the
        #            "Other flows" rows 58/59) -- this is the number to use
        #   96 = US and foreign, 98 = foreign only, 57/58/59 = "other flows" pools
        # A row whose origin equals its destination is the NON-MIGRANT row (filers
        # who stayed put); it dwarfs every real flow, so it is split out separately.
        is_total = (other_state == 97) & (other_county == "000")
        stayed = other_fips == df["fips"]

        if is_total.any():
            g = df[is_total].groupby("fips").agg(returns=("n1", "sum"), agi=("agi", "sum"))
            how = "summary rows"
        else:   # fallback for layouts without the 97/000 row: sum named flows + pools
            pooled = other_state.isin([58, 59]) & (other_county == "000")
            named = other_state.between(1, 56) & (other_county != "000") & ~stayed
            g = df[named | pooled].groupby("fips").agg(returns=("n1", "sum"), agi=("agi", "sum"))
            how = "summed flows"

        out = pd.DataFrame({
            "fips": g.index,
            f"irs_{kind}_returns": g["returns"].values,
            # AGI is in $1,000s; x1000 gives average AGI per household that moved
            f"irs_{kind}_agi_per_return": (1000 * g["agi"] / g["returns"]).replace(
                [np.inf, -np.inf], np.nan).values,
        })
        if kind == "in":        # identical in both files, so read it once
            nonmig = df[stayed].groupby("fips")["n1"].sum().rename("irs_nonmigrant_returns")
            out = out.merge(nonmig.reset_index(), on="fips", how="left")
        frames[kind] = out
        log(f"  [IRS] {path.name}: {len(out):,} counties ({kind}flow, {how}), "
            f"{int(df.loc[stayed, 'n1'].sum()):,} non-migrant returns set aside")

    if not frames:
        return None, []
    if len(frames) == 1:
        log(f"  [IRS] WARNING: only the {list(frames)[0]}flow file was read; "
            "net migration needs both files")

    merged = None
    for part in frames.values():
        merged = part if merged is None else merged.merge(part, on="fips", how="outer")

    descriptions = {
        "irs_in_returns": "Tax returns moving INTO the county from other US counties (households)",
        "irs_out_returns": "Tax returns moving OUT of the county to other US counties (households)",
        "irs_nonmigrant_returns": "Tax returns that stayed in the same county (non-migrants)",
        "irs_in_agi_per_return": "Average AGI ($) per household moving in",
        "irs_out_agi_per_return": "Average AGI ($) per household moving out",
    }
    entries = [{"column": c, "source": "IRS SOI county migration", "description": descriptions[c]}
               for c in merged.columns if c != "fips"]
    return merged, entries

def add_migration_rates(df: pd.DataFrame, dictionary: list) -> tuple:
    """Rates are built after counties are combined, so numerator and denominator match."""
    if "irs_in_returns" not in df or "acs_total_pop" not in df:
        return df, dictionary
    pop = df["acs_total_pop"].where(df["acs_total_pop"] > 0)
    specs = {
        "irs_in_rate_per_1000": (1000 * df["irs_in_returns"] / pop,
                                 "Households moving in per 1,000 residents"),
        "irs_out_rate_per_1000": (1000 * df.get("irs_out_returns", np.nan) / pop,
                                  "Households moving out per 1,000 residents"),
    }
    if "irs_out_returns" in df:
        specs["irs_net_rate_per_1000"] = (
            1000 * (df["irs_in_returns"] - df["irs_out_returns"]) / pop,
            "Net migration per 1,000 residents (in minus out)")
        specs["irs_churn_rate_per_1000"] = (
            1000 * (df["irs_in_returns"] + df["irs_out_returns"]) / pop,
            "Total moves per 1,000 residents (in plus out): population turnover")
    if "irs_nonmigrant_returns" in df and "irs_out_returns" in df:
        movers_plus_stayers = df["irs_nonmigrant_returns"] + df["irs_out_returns"]
        specs["irs_pct_stayed"] = (
            100 * df["irs_nonmigrant_returns"] / movers_plus_stayers.replace(0, np.nan),
            "% of resident filers who stayed in the county (rootedness)")
    if "irs_in_agi_per_return" in df and "irs_out_agi_per_return" in df:
        specs["irs_agi_in_out_ratio"] = (
            df["irs_in_agi_per_return"] / df["irs_out_agi_per_return"].replace(0, np.nan),
            "Income of arrivers vs. leavers (>1 = in-movers earn more)")
    for name, (values, desc) in specs.items():
        df[name] = pd.Series(values, index=df.index).replace([np.inf, -np.inf], np.nan)
        dictionary.append({"column": name, "source": "Derived from IRS + ACS", "description": desc})
    log(f"  Added {len(specs)} migration rate features")
    return df, dictionary


# ---------------------------------------------------------------------------
# Social Capital Atlas (Opportunity Insights / Meta)
# ---------------------------------------------------------------------------
SCAP_VARS = {
    "ec_county": ("scap_economic_connectedness",
                  "Economic connectedness: share of high-income friends among low-income residents"),
    "child_ec_county": ("scap_child_economic_connectedness",
                        "Economic connectedness among children/young people"),
    "ec_grp_mem_county": ("scap_ec_group_membership",
                          "Economic connectedness within groups (clubs, organizations)"),
    "exposure_grp_mem_county": ("scap_exposure_to_high_income",
                                "Exposure of low-income residents to high-income people in groups"),
    "bias_grp_mem_county": ("scap_friending_bias",
                            "Friending bias: tendency to befriend similar-income people"),
    "clustering_county": ("scap_clustering",
                          "Cohesiveness: share of a person's friends who are also friends"),
    "support_ratio_county": ("scap_support_ratio",
                             "Share of friendships supported by a mutual friend"),
    "volunteering_rate_county": ("scap_volunteering_rate",
                                 "Share of members of volunteering/activism groups"),
    "civic_organizations_county": ("scap_civic_organizations",
                                   "Civic organizations per 1,000 users"),
}


def load_social_capital(folder: Path):
    files = find_files(folder, (".csv",))
    if not files:
        log(f"  [SCAP] no CSV in {folder}; skipping")
        return None, []
    path = next((f for f in files if "county" in f.name.lower()), files[0])
    df = read_table(path)
    lower = {c.lower(): c for c in df.columns}

    fips_col = lower.get("county") or lower.get("county_fips") or lower.get("fips")
    if fips_col is None:
        log(f"  [SCAP] no county id column in {path.name}. Columns: {list(df.columns)[:15]}")
        return None, []

    out = pd.DataFrame({"fips": clean_fips(df[fips_col])})
    entries, missing = [], []
    for src, (name, desc) in SCAP_VARS.items():
        if src in lower:
            out[name] = pd.to_numeric(df[lower[src]], errors="coerce").values
            entries.append({"column": name, "source": f"Social Capital Atlas ({src})", "description": desc})
        else:
            missing.append(src)
    if missing:
        log(f"  [SCAP] not found in this file (names vary by version): {missing}")
    out = out[out["fips"].map(is_county_fips)]
    if len(entries) == 0:
        log("  [SCAP] no usable measures found; skipping")
        return None, []
    log(f"  [SCAP] {len(out):,} counties, {len(entries)} measures")
    return out, entries


# ---------------------------------------------------------------------------
# IHME combined units
# ---------------------------------------------------------------------------
def combine_ihme_units(df: pd.DataFrame) -> pd.DataFrame:
    """
    IHME reports some groups of counties as one unit (e.g. Virginia independent
    cities with their surrounding county), so every member gets identical outcome
    values. This collapses each group to one row:
      - predictors: population-weighted mean (counts are summed; RUCC comes
        from the largest county)
      - fips: the largest county in the group
      - member_fips: all counties in the group, separated by ';'
    Rows with no ACS population (obsolete county codes) are dropped first.
    """
    df = df.copy()
    has_pop = "acs_total_pop" in df

    if has_pop:
        obsolete = df["acs_total_pop"].isna()
        if obsolete.any():
            log(f"  Dropping {obsolete.sum()} counties with no ACS data "
                f"(obsolete codes): {df.loc[obsolete, 'fips'].tolist()}")
            df = df[~obsolete]

    outs = [c for c in OUTCOME_COLS if c in df]
    key = df[outs].round(6).astype(str).agg("|".join, axis=1)
    size = key.map(key.value_counts())

    singles = df[size == 1].copy()
    singles["member_fips"] = singles["fips"]
    multi = df[size > 1]
    if multi.empty:
        return singles

    skip = set(outs) | {"fips", "state_fips", "state", "census_division"}
    num_cols = [c for c in df.columns if c not in skip and pd.api.types.is_numeric_dtype(df[c])]

    rows = []
    for _, g in multi.groupby(key[size > 1]):
        w = g["acs_total_pop"] if has_pop else pd.Series(1.0, index=g.index)
        lead = g.loc[w.idxmax()]
        row = lead.to_dict()
        for c in num_cols:
            if c in LARGEST_COLS:
                continue
            if c in SUM_COLS:
                row[c] = g[c].sum(min_count=1)
                continue
            ok = g[c].notna()
            row[c] = np.average(g.loc[ok, c], weights=w[ok]) if ok.any() and w[ok].sum() > 0 else np.nan
        if has_pop and "acs_log_pop" in row:
            row["acs_log_pop"] = np.log(row["acs_total_pop"])
        row["member_fips"] = ";".join(sorted(g["fips"]))
        rows.append(row)

    combined = pd.DataFrame(rows)
    log(f"  Combined {len(multi)} counties into {len(combined)} IHME units "
        f"(e.g. {combined['member_fips'].iloc[0]})")
    out = pd.concat([singles, combined], ignore_index=True).sort_values("fips")
    out[num_cols] = out[num_cols].apply(pd.to_numeric, errors="coerce")
    return out.reset_index(drop=True)


# ---------------------------------------------------------------------------
# Build
# ---------------------------------------------------------------------------
def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser(description="Build county life expectancy dataset")
    ap.add_argument("--raw-dir", default="data/raw")
    ap.add_argument("--out-dir", default="data/processed")
    # parse_known_args ignores extra arguments Jupyter adds (e.g. "-f kernel.json")
    args, _ = ap.parse_known_args(argv)
    raw, out_dir = Path(args.raw_dir), Path(args.out_dir)

    log("Loading outcome (IHME)...")
    base, dictionary = load_ihme(raw / "ihme")
    base = harmonize_fips(base, "IHME")

    loaders = [
        ("CHR", lambda: load_chr(raw / "chr")),
        ("ACS", lambda: load_acs(raw / "acs" / f"acs5_{ACS_YEAR}_county.csv")),
        ("RUCC", lambda: load_rucc(raw / "rucc")),
        ("CMS", lambda: load_cms(raw / "cms")),
        ("IRS", lambda: load_irs(raw / "irs")),
        ("SCAP", lambda: load_social_capital(raw / "scap")),
    ]
    for name, loader in loaders:
        log(f"Loading {name}...")
        df, entries = loader()
        if df is None:
            continue
        df = harmonize_fips(df, name)
        matched = base["fips"].isin(df["fips"])
        unmatched = base.loc[~matched, "fips"].tolist()
        log(f"  [{name}] matched {matched.sum():,} of {len(base):,} outcome counties ({matched.mean():.1%})")
        if unmatched:
            log(f"  [{name}] unmatched FIPS (first 10): {unmatched[:10]}")
        base = base.merge(df, on="fips", how="left", validate="one_to_one")
        dictionary += entries

    log("Cleaning county codes...")
    base = combine_ihme_units(base)
    base, dictionary = add_migration_rates(base, dictionary)

    # Geography columns for mapping and spatial cross-validation
    base.insert(1, "state_fips", base["fips"].str[:2])
    base.insert(2, "state", base["state_fips"].map(STATE_ABBR))
    base.insert(3, "census_division", base["state"].map(STATE_TO_DIVISION))
    member = base.pop("member_fips")
    base.insert(4, "member_fips", member)
    dictionary = [
        {"column": "fips", "source": "Key", "description": "5-digit county FIPS code (largest county if a combined unit)"},
        {"column": "state_fips", "source": "Derived", "description": "2-digit state FIPS code"},
        {"column": "state", "source": "Derived", "description": "State abbreviation"},
        {"column": "census_division", "source": "Derived", "description": "Census division (use for spatial CV)"},
        {"column": "member_fips", "source": "Derived",
         "description": "Counties in this row; several (';'-separated) when IHME combined them into one unit"},
    ] + dictionary

    out_dir.mkdir(parents=True, exist_ok=True)
    base.to_csv(out_dir / "county_le50_2019.csv", index=False)
    pd.DataFrame(dictionary).to_csv(out_dir / "data_dictionary.csv", index=False)
    miss = base.isna().mean().sort_values(ascending=False).rename("share_missing")
    miss.to_frame().rename_axis("column").to_csv(out_dir / "missingness_report.csv")

    log("\nDone.")
    log(f"  Rows (counties): {len(base):,}")
    log(f"  Columns: {base.shape[1]}")
    log(f"  Outcome le50_both: mean {base['le50_both'].mean():.2f}, "
        f"range {base['le50_both'].min():.2f}-{base['le50_both'].max():.2f}")
    high_missing = miss[miss > 0.2]
    if len(high_missing):
        log(f"  Columns more than 20% missing (consider dropping): {list(high_missing.index)}")
    log(f"  Files written to {out_dir.resolve()}")


if __name__ == "__main__":
    main()


Loading outcome (IHME)...
  [IHME] 3,127 counties; outcomes: ['le50_both', 'le50_male', 'le50_female', 'le0_both']
Loading CHR...
  [CHR] 3,142 counties, 79 measures kept
  [CHR] dropped 6 death-based measures to avoid leakage: ['Premature Death raw value', 'Alcohol-Impaired Driving Deaths raw value', 'Life Expectancy raw value', 'Premature Age-Adjusted Mortality raw value', 'Child Mortality raw value', 'Infant Mortality raw value']
  [CHR] kept 6 cause-specific death measures by choice: ['chr_injury_deaths', 'chr_drug_overdose_deaths', 'chr_homicides', 'chr_suicides', 'chr_firearm_fatalities', 'chr_motor_vehicle_crash_deaths']
  [CHR] matched 3,111 of 3,118 outcome counties (99.8%)
  [CHR] unmatched FIPS (first 10): ['02010', '02140', '02201', '02231', '02232', '02280', '30113']
Loading ACS...
  [ACS] using cached file data\raw\acs\acs5_2019_county.csv
  [ACS] 3,142 counties, 16 variables
  [ACS] matched 3,111 of 3,118 outcome counties (99.8%)
  [ACS] unmatched FIPS (first 10): ['0201

## Validation: outcome sanity checks

In [14]:
import pandas as pd
df = pd.read_csv(r"C:\Users\14438\data\processed\county_le50_2019.csv",
                 dtype={"fips": str, "state_fips": str})
cols = ["fips", "state", "le50_both", "le50_male", "le50_female", "le0_both"]

# 1. Highest and lowest counties
print(df.nlargest(10, "le50_both")[cols], "\n")
print(df.nsmallest(10, "le50_both")[cols], "\n")

# 2. Both-sexes value should fall between the male and female values
lo = df[["le50_male", "le50_female"]].min(axis=1)
hi = df[["le50_male", "le50_female"]].max(axis=1)
print("Counties outside male-female range:", ((df.le50_both < lo - 0.1) | (df.le50_both > hi + 0.1)).sum())

# 3. Birth vs. age 50: gap should be roughly 44-50 years, never above 50
print("Correlation:", round(df.le50_both.corr(df.le0_both), 3))
print((df.le0_both - df.le50_both).describe())


       fips state  le50_both  le50_male  le50_female   le0_both
286   08117    CO  44.535222  43.380356    45.550996  91.722415
277   08097    CO  43.783417  42.145239    45.378115  91.342130
250   08037    CO  40.963863  39.633218    42.340948  88.472717
284   08113    CO  39.152954  37.776252    40.720432  86.247958
312   12021    FL  38.310691  36.246193    40.311089  85.258260
67    02016    AK  38.271259  38.992336    37.042110  85.682486
202   06051    CA  38.238441  36.906297    39.756262  85.657943
2664  48377    TX  37.928869  36.055402    39.968127  85.148763
3075  56039    WY  37.923461  37.132964    38.721971  85.905597
73    02100    AK  37.494638  34.623862    40.446980  83.943410 

       fips state  le50_both  le50_male  le50_female   le0_both
2876  51690    VA  22.855695  20.807242    24.765437  67.928609
364   12125    FL  22.943556  20.284234    29.681926  67.738851
2871  51640    VA  24.086089  21.518567    26.573489  69.941962
1069  21193    KY  24.327319  22.82812

## Validation: confirm cleaning steps,

In [15]:
out = ["le50_both", "le50_male", "le50_female", "le0_both"]

# Counties sharing identical outcome values (IHME combined units)
combined = df[df.duplicated(out, keep=False)].sort_values(out)
print(combined[["fips", "state"] + out + ["acs_total_pop"]])

# Counties with no ACS data (likely obsolete codes)
print(df.loc[df["acs_total_pop"].isna(), ["fips", "state", "le50_both"]])

# The 2 counties where both-sexes falls outside the male-female range
lo = df[["le50_male", "le50_female"]].min(axis=1)
hi = df[["le50_male", "le50_female"]].max(axis=1)
print(df.loc[(df.le50_both < lo - 0.1) | (df.le50_both > hi + 0.1), cols])

Empty DataFrame
Columns: [fips, state, le50_both, le50_male, le50_female, le0_both, acs_total_pop]
Index: []
Empty DataFrame
Columns: [fips, state, le50_both]
Index: []
       fips state  le50_both  le50_male  le50_female   le0_both
70    02060    AK  33.077225  31.150506          NaN  77.388910
2062  39121    OH  34.542615  34.272996    34.280094  80.925045


## ## Feature coverage

Lists every feature with its source, description, and percent of counties filled, then
saves the table to `feature_coverage.xlsx` for reference.

Coverage ratings: **Good** (95%+) ready to use, **OK** (80–95%) usable with imputation
for linear models, **Low** (<80%) candidates to drop. Missing values are not random —
most come from CDC suppressing small counts in small, rural counties — so features with
low coverage would bias the model toward larger counties.

In [16]:
import pandas as pd

path = r"C:\Users\14438\data\processed"
df = pd.read_csv(path + r"\county_le50_2019.csv",
                 dtype={"fips": str, "state_fips": str, "member_fips": str})
dd = pd.read_csv(path + r"\data_dictionary.csv")

# Everything except IDs and outcomes is a feature
id_cols = ["fips", "state_fips", "state", "census_division", "member_fips"]
outcomes = ["le50_both", "le50_male", "le50_female", "le0_both"]
features = [c for c in df.columns if c not in id_cols + outcomes]

summary = pd.DataFrame({
    "feature": features,
    "pct_filled": (df[features].notna().mean() * 100).round(1).values,
    "n_missing": df[features].isna().sum().values,
})
summary = summary.merge(dd.rename(columns={"column": "feature"}), on="feature", how="left")

# Shorter labels for readability
summary["source"] = summary["source"].str.split().str[0].replace({"Census": "ACS"})
summary["description"] = summary["description"].str.replace(" raw value", "", case=False)

# Coverage rating
summary["coverage"] = pd.cut(summary["pct_filled"], bins=[-1, 80, 95, 100],
                             labels=["Low (<80%)", "OK (80-95%)", "Good (95%+)"])

summary = summary.sort_values(["source", "pct_filled"], ascending=[True, False])
summary = summary[["source", "feature", "description", "pct_filled", "n_missing", "coverage"]]

# Show everything
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
print(f"{len(features)} features across {len(df):,} counties\n")
print(summary.to_string(index=False))
print("\n", summary["coverage"].value_counts().sort_index())

# Save for reference
summary.to_excel(path + r"\feature_coverage.xlsx", index=False)

123 features across 3,079 counties

 source                                           feature                                                                               description  pct_filled  n_missing    coverage
    ACS                                     acs_total_pop                                                                          Total population       100.0          0 Good (95%+)
    ACS                                       acs_log_pop                                                                   Log of total population       100.0          0 Good (95%+)
    ACS                                    acs_median_age                                                                                Median age       100.0          0 Good (95%+)
    ACS                              acs_median_hh_income                                                          Median household income (2019 $)       100.0          0 Good (95%+)
    ACS                                          

## Check fo counties with no migraion data

In [18]:
cols = ["irs_in_rate_per_1000", "irs_out_rate_per_1000", "irs_net_rate_per_1000",
        "irs_churn_rate_per_1000", "irs_pct_stayed", "irs_agi_in_out_ratio"]
print(df[cols].describe().round(1).T)
print("\nCounties with no migration data:")
print(df.loc[df["irs_in_returns"].isna(), ["fips", "state", "acs_total_pop"]].head(10))

                          count  mean   std   min   25%   50%   75%    max
irs_in_rate_per_1000     3051.0  23.1   8.7   5.3  17.1  21.5  27.5   78.7
irs_out_rate_per_1000    3068.0  23.2   7.2   8.7  18.6  22.2  26.3  114.9
irs_net_rate_per_1000    3049.0  -0.1   4.8 -92.2  -2.6  -0.5   2.1   36.8
irs_churn_rate_per_1000  3049.0  46.4  15.2  16.0  36.3  43.8  53.9  176.1
irs_pct_stayed           3049.0  93.6   1.8  65.3  92.9  93.9  94.7   97.7
irs_agi_in_out_ratio     3049.0   1.0   0.2  -5.0   0.9   1.0   1.1    3.2

Counties with no migration data:
       fips state  acs_total_pop
260   08061    CO         1489.0
884   20033    KS         1740.0
903   20071    KS         1185.0
905   20075    KS         2591.0
1407  28055    MS         1361.0
1582  30011    MT         1331.0
1586  30019    MT         1730.0
1593  30033    MT         1036.0
1628  30109    MT         1122.0
1634  31015    NE         1937.0


## Correction: negative AGI in the migration income ratio

irs_agi_in_out_ratio compares the average income of households moving in to those
moving out. IRS aggregate AGI can be negative in counties where business losses exceed
reported income, which makes the ratio meaningless (it came through as low as −5.0).
Those counties are set to missing rather than kept as extreme values, and the corrected
file is written back to disk.

In [22]:
import numpy as np

# Blank the income ratio where either side has negative average AGI
bad = (df["irs_in_agi_per_return"] <= 0) | (df["irs_out_agi_per_return"] <= 0)
print("Counties fixed:", bad.sum())
df.loc[bad, "irs_agi_in_out_ratio"] = np.nan

# Save the corrected file back
df.to_csv(path + r"\county_le50_2019.csv", index=False)
print("Saved. Ratio range now:",
      df["irs_agi_in_out_ratio"].min().round(2), "to",
      df["irs_agi_in_out_ratio"].max().round(2))

Counties fixed: 1
Saved. Ratio range now: 0.05 to 3.22
